In [1]:
import os, glob, inspect
import torch
import numpy as np
import pandas as pd
import nugraph as ng

DATA_PATH = "/home/apaudel/NuGraph/scripts/merged1_Aug6_40k_pmtpmt_g4lepdir_with_vpred_nexusonly"
RUN_NAME = "merged1_Aug20_40k_g4lepdir_tpc_only_semantic_vertex_direction_fixedvpred_nexusfeat_ep100_bs32_in8_sp5"

ckpts = glob.glob(f"/home/apaudel/NuGraph/logs/{RUN_NAME}/**/*.ckpt", recursive=True)
assert ckpts, f"No checkpoint found for {RUN_NAME}"
CKPT = max(ckpts, key=os.path.getmtime)

print("DATA_PATH =", DATA_PATH)
print("RUN_NAME  =", RUN_NAME)
print("CKPT      =", CKPT)

raw = torch.load(CKPT, map_location="cpu")
hp = raw.get("hyper_parameters", {})

print("\nCheckpoint hparams:")
for k in ["semantic_head", "vertex_head", "direction_head", "event_head", "sp_features", "use_optical", "use_pmt_pmt"]:
    print(f"{k:16s} =", hp.get(k, "MISSING"))

Data = ng.data.H5DataModule
Model = ng.models.NuGraph3

nudata = Data(
    DATA_PATH,
    batch_size=32,
    num_workers=0,
    model=Model,
    shuffle="random",
    featext3d=True,
)

model = Model.load_from_checkpoint(CKPT)
model.eval()

print("\nDataset lengths:")
print("train =", len(nudata.train_dataset))
print("val   =", len(nudata.val_dataset))
print("test  =", len(nudata.test_dataset))

print("\nModel sanity:")
print("encoder.sp_net is None =", model.encoder.sp_net is None)
print("has vertex_decoder     =", hasattr(model, "vertex_decoder"))
print("has direction_decoder  =", hasattr(model, "direction_decoder"))

print("\nDirection decoder source check:")
src = inspect.getsource(model.direction_decoder.__class__)
for line in src.splitlines():
    if "USE_TRUE_VERTEX_REFERENCE" in line or "v_pred" in line or "y_vtx" in line or "return evt" in line:
        print(line)

assert "v_pred" in src, "direction.py source does not contain v_pred"
assert "return evt.v_pred" in src or "return evt[\"v_pred\"]" in src, "direction decoder may not be using evt.v_pred"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

batch = next(iter(nudata.test_dataloader()))
print("\nBefore forward evt keys:")
print(list(batch["evt"].keys()))

batch = batch.to(device)

with torch.no_grad():
    model(batch)

print("\nAfter forward evt keys:")
print(list(batch["evt"].keys()))

evt = batch["evt"]

print("\nImportant field shapes:")
for k in list(evt.keys()):
    x = getattr(evt, k)
    if torch.is_tensor(x):
        print(f"{k:15s}", tuple(x.shape), x.dtype)

for name in ["y_vtx", "v_pred", "v", "y_dir", "d"]:
    if hasattr(evt, name):
        x = getattr(evt, name).detach().cpu()
        print(f"\n{name}:")
        print("  shape =", tuple(x.shape))
        print("  first 5 =")
        print(x[:5])
        if x.ndim == 2 and x.shape[1] == 3:
            print("  norm first 5 =", torch.linalg.norm(x[:5].float(), dim=1))
            print("  norm min/mean/max =", 
                  torch.linalg.norm(x.float(), dim=1).min().item(),
                  torch.linalg.norm(x.float(), dim=1).mean().item(),
                  torch.linalg.norm(x.float(), dim=1).max().item())

DATA_PATH = /home/apaudel/NuGraph/scripts/merged1_Aug6_40k_pmtpmt_g4lepdir_with_vpred_nexusonly
RUN_NAME  = merged1_Aug20_40k_g4lepdir_tpc_only_semantic_vertex_direction_fixedvpred_nexusfeat_ep100_bs32_in8_sp5
CKPT      = /home/apaudel/NuGraph/logs/merged1_Aug20_40k_g4lepdir_tpc_only_semantic_vertex_direction_fixedvpred_nexusfeat_ep100_bs32_in8_sp5/version_1/checkpoints/epoch=99-step=94000.ckpt

Checkpoint hparams:
semantic_head    = True
vertex_head      = True
direction_head   = True
event_head       = False
sp_features      = 5
use_optical      = False
use_pmt_pmt      = False

Dataset lengths:
train = 30080
val   = 1672
test  = 1672

Model sanity:
encoder.sp_net is None = False
has vertex_decoder     = True
has direction_decoder  = True

Direction decoder source check:
      USE_TRUE_VERTEX_REFERENCE = True
    This uses evt.v_pred as the reference point. That is intentional for the
      USE_TRUE_VERTEX_REFERENCE = True
    USE_TRUE_VERTEX_REFERENCE = True
        if self.USE_TRUE

In [2]:
import numpy as np
import pandas as pd

true_d = df[["true_dx", "true_dy", "true_dz"]].values
pred_d = df[["pred_dx", "pred_dy", "pred_dz"]].values

def unit(x):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(n, 1e-12, None)

true_u = unit(true_d)
pred_u = unit(pred_d)

cosang = np.sum(true_u * pred_u, axis=1)
cosang = np.clip(cosang, -1, 1)

angle_deg = np.degrees(np.arccos(cosang))
axis_angle_deg = np.degrees(np.arccos(np.abs(cosang)))

summary = {
    "N": len(cosang),
    "cos mean": np.mean(cosang),
    "cos median": np.median(cosang),
    "angle median deg": np.median(angle_deg),
    "angle mean deg": np.mean(angle_deg),
    "angle p68 deg": np.percentile(angle_deg, 68),
    "angle p90 deg": np.percentile(angle_deg, 90),
    "axis angle median deg": np.median(axis_angle_deg),
    "frac cos > 0.8": np.mean(cosang > 0.8),
    "frac cos > 0.9": np.mean(cosang > 0.9),
    "frac cos > 0.95": np.mean(cosang > 0.95),
    "frac cos < -0.8": np.mean(cosang < -0.8),
}

display(pd.DataFrame([summary]).round(4))

for i, name in enumerate(["x", "y", "z"]):
    corr = np.corrcoef(true_u[:, i], pred_u[:, i])[0, 1]
    print(f"component {name} correlation = {corr:.4f}")

NameError: name 'df' is not defined